In [1]:
import torch
import clip
import laion_clap
from transformers import AutoModelForCausalLM
import warnings

print(f"PyTorch 버전: {torch.__version__}")
print(f"GPU 사용 가능 여부: {torch.cuda.is_available()}")

# CLIP 로드 테스트
model, preprocess = clip.load("ViT-B/32", device="cuda")
print("CLIP 로드 성공!")

# CLAP 로드 테스트
clap_model = laion_clap.CLAP_Module(enable_fusion=False)
print("CLAP 모듈 로드 성공!")

# 1. 파이썬 수준의 경고 숨기기
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# 2. 오디오 라이브러리(audioread) 관련 로깅 레벨 조정
import logging
logging.getLogger('librosa').setLevel(logging.ERROR)

/home/oem/anaconda3/envs/qx_llm/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


PyTorch 버전: 2.10.0+cu128
GPU 사용 가능 여부: True
CLIP 로드 성공!


/home/oem/anaconda3/envs/qx_llm/lib/python3.10/site-packages/librosa/core/intervals.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
/home/oem/anaconda3/envs/qx_llm/lib/python3.10/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


CLAP 모듈 로드 성공!


In [2]:
import h5py
import torch
from torch.utils.data import Dataset

class H5PersonalityDataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = h5_path
        self.dataset = None  # Worker 프로세스별로 핸들을 따로 관리하기 위함
        
        # 전체 길이는 미리 파악
        with h5py.File(self.h5_path, 'r') as f:
            self.length = len(f['label'])

    def __len__(self):
        return self.length

    def _get_h5_file(self):
        """Worker 프로세스마다 하나의 파일 핸들을 유지하도록 함"""
        if self.dataset is None:
            self.dataset = h5py.File(self.h5_path, 'r', swmr=True)
        return self.dataset

    def __getitem__(self, idx):
        f = self._get_h5_file()
        
        # 1. 수치 데이터 로드 및 텐서 변환
        video = torch.from_numpy(f['video'][idx])
        audio = torch.from_numpy(f['audio'][idx])
        label = torch.from_numpy(f['label'][idx])/120
        
        # 2. 텍스트 데이터 로드 (transcription)
        # H5의 가변길이 문자열은 읽어온 후 디코딩이 필요할 수 있습니다.
        transcription = f['transcription'][idx]
        if isinstance(transcription, bytes):
            transcription = transcription.decode('utf-8')
            
        return {
            "video": video,
            "audio": audio,
            "label": label,
            "transcription": transcription
        }

    def __del__(self):
        """객체 소멸 시 파일 핸들 닫기"""
        if self.dataset is not None:
            self.dataset.close()

In [3]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer

class LatentCoTPersonalityLLM(nn.Module):
    def __init__(
        self,
        gemma_model_id="beomi/gemma-ko-7b",  # ✅ 改名更清晰
        clip_dim=768,
        clap_dim=512,
        num_query_tokens=4,
        device="cuda",
        device_map=None,  # ✅ 新增：可控单卡/多卡
    ):
        super().__init__()
        self.device = device
        self.num_query_tokens = num_query_tokens

        # 1. Core LLM & Tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(gemma_model_id)

        # ✅ device_map 不要传 "cuda" 字符串；默认单卡最稳
        if device_map is None:
            device_map = {"": 3} if torch.cuda.is_available() else None

        self.llama = AutoModelForCausalLM.from_pretrained(
            gemma_model_id,
            torch_dtype=torch.bfloat16,
            device_map=device_map
        )

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        if getattr(self.llama.config, "pad_token_id", None) is None:
            self.llama.config.pad_token_id = self.tokenizer.eos_token_id

        self.special_tokens = ["<V_START>", "<V_END>", "<A_START>", "<A_END>"]
        self.tokenizer.add_special_tokens({"additional_special_tokens": self.special_tokens})
        self.llama.resize_token_embeddings(len(self.tokenizer))

        if hasattr(self.llama.config, "hidden_size"):
            self.llm_dim = self.llama.config.hidden_size
        elif hasattr(self.llama.config, "text_config") and hasattr(self.llama.config.text_config, "hidden_size"):
            self.llm_dim = self.llama.config.text_config.hidden_size
        else:
            raise AttributeError("Cannot find hidden_size in model config.")

        # 2. Latent Query Tokens (추론 병목)
        self.query_tokens = nn.Parameter(torch.zeros(1, num_query_tokens, self.llm_dim))
        nn.init.trunc_normal_(self.query_tokens, std=0.02)

        # 3. Semantic Projectors
        self.video_projector = nn.Sequential(
            nn.LayerNorm(clip_dim),
            nn.Linear(clip_dim, self.llm_dim),
            nn.GELU(),
            nn.Linear(self.llm_dim, self.llm_dim),
            nn.LayerNorm(self.llm_dim)
        )

        self.audio_projector = nn.Sequential(
            nn.LayerNorm(clap_dim),
            nn.Linear(clap_dim, self.llm_dim),
            nn.GELU(),
            nn.Linear(self.llm_dim, self.llm_dim),
            nn.LayerNorm(self.llm_dim)
        )

        # 4. Regression Head
        self.regression_head = nn.Sequential(
            nn.LayerNorm(self.llm_dim),
            nn.Linear(self.llm_dim, 512),
            nn.SiLU(),
            nn.Dropout(0.1),
            nn.Linear(512, 5)
            # nn.Sigmoid()
        )

        self._init_projector_weights()
        self.video_projector = self.video_projector.to(torch.bfloat16)
        self.audio_projector = self.audio_projector.to(torch.bfloat16)
        self.regression_head = self.regression_head.float()

    def _init_projector_weights(self):
        for m in [self.video_projector, self.audio_projector]:
            for layer in m:
                if isinstance(layer, nn.Linear):
                    nn.init.xavier_uniform_(layer.weight, gain=0.01)
                    if layer.bias is not None:
                        nn.init.zeros_(layer.bias)
                elif isinstance(layer, nn.LayerNorm):
                    layer.eps = 1e-4

    def forward(self, input_ids, attention_mask, video_features, audio_features):
        target_dtype = self.llama.dtype
        batch_size = video_features.shape[0]

        # ✅ 关键：所有输入/新建tensor跟着 embedding device，避免 cuda:1/cuda:2 混用
        embed_device = self.llama.get_input_embeddings().weight.device

        input_ids = input_ids.to(embed_device)
        attention_mask = attention_mask.to(embed_device)
        video_features = video_features.to(embed_device)
        audio_features = audio_features.to(embed_device)

        # 1. 멀티모달 특징 투영
        v_norm = torch.nn.functional.normalize(video_features.float(), p=2, dim=-1).to(target_dtype)
        a_norm = torch.nn.functional.normalize(audio_features.float(), p=2, dim=-1).to(target_dtype)

        v_proj = self.video_projector(v_norm)
        if v_proj.dim() == 2:
            v_proj = v_proj.unsqueeze(1)

        a_proj = self.audio_projector(a_norm)
        if a_proj.dim() == 2:
            a_proj = a_proj.unsqueeze(1)

        # 2. 스페셜 토큰 임베딩
        special_token_ids = self.tokenizer.convert_tokens_to_ids(self.special_tokens)
        special_token_tensor = torch.tensor(special_token_ids, device=embed_device)
        special_embeds = self.llama.get_input_embeddings()(special_token_tensor)

        def prepare_embed(embed):
            return embed.reshape(1, 1, -1).expand(batch_size, -1, -1)

        v_start_embed = prepare_embed(special_embeds[0])
        v_end_embed   = prepare_embed(special_embeds[1])
        a_start_embed = prepare_embed(special_embeds[2])
        a_end_embed   = prepare_embed(special_embeds[3])

        latent_queries = self.query_tokens.expand(batch_size, -1, -1).to(target_dtype)  # 원래 주석 유지

        text_embeds = self.llama.get_input_embeddings()(input_ids)

        full_embeds = torch.cat([
            text_embeds,
            v_start_embed, v_proj, v_end_embed,
            a_start_embed, a_proj, a_end_embed,
            latent_queries
        ], dim=1)

        num_extra_tokens = full_embeds.shape[1] - attention_mask.shape[1]
        extra_mask = torch.ones(batch_size, num_extra_tokens, device=embed_device, dtype=attention_mask.dtype)
        full_attention_mask = torch.cat([attention_mask, extra_mask], dim=1)

        B, L, _ = full_embeds.shape
        token_type_ids = torch.zeros((B, L), device=full_embeds.device, dtype=torch.long)

        outputs = self.llama(
            inputs_embeds=full_embeds,
            attention_mask=full_attention_mask,
            token_type_ids=token_type_ids,
            output_hidden_states=True,
            return_dict=True
        )
        
        query_start_idx = full_embeds.shape[1] - self.num_query_tokens
        query_states = outputs.hidden_states[-1][:, query_start_idx:, :]

        # aggregation
        z = query_states.mean(dim=1)

        # prediction
        pred_scores = self.regression_head(z.float())
        
        return pred_scores


In [4]:
import os
import torch
import torch.nn as nn
import numpy as np
import clip
import laion_clap
from torch.utils.data import DataLoader
from tqdm import tqdm
from peft import LoraConfig, get_peft_model

os.environ["TOKENIZERS_PARALLELISM"] = "false"

device = "cuda:3" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 4
EPOCHS = 30
LR = 1e-6
TRAIN_H5 = "./keti_scene3_train.h5"
VAL_H5 = "./keti_scene3_val.h5"

print("🚀 Latent CoT Personality (Gemma) 모델 초기화 중...")

model = LatentCoTPersonalityLLM(
    gemma_model_id="hyokwan/kopo_gemma3_4b_fintech",
    device=device,
    device_map={"": 3} if torch.cuda.is_available() else None
).to(device)

clip_model, _ = clip.load("ViT-L/14", device=device)
clap_model = laion_clap.CLAP_Module(enable_fusion=False).to(device)
clap_model.load_ckpt()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    modules_to_save=[
        "embed_tokens",
        "video_projector",
        "audio_projector",
        "regression_head"
    ]
)

model.llama = get_peft_model(model.llama, lora_config)
model.llama.print_trainable_parameters()

train_dataset = H5PersonalityDataset(TRAIN_H5)
val_dataset = H5PersonalityDataset(VAL_H5)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

# ✅ MAE loss
criterion = nn.L1Loss()

history = {"train_mae": [], "val_mae": [], "train_1mae": [], "val_1mae": []}
best_val_1mae = -float("inf")

for epoch in range(EPOCHS):
    # =========================
    # Train
    # =========================
    model.train()
    train_loss_sum = 0.0

    pbar = tqdm(train_loader, desc=f"🔥 [Latent-CoT|Gemma] Ep {epoch+1}/{EPOCHS}")
    for batch in pbar:
        videos = batch["video"].to(device)
        audios = batch["audio"].to(device).float()
        target_labels = batch["label"].to(device).float()
        transcriptions = batch["transcription"]
        b = videos.shape[0]

        with torch.no_grad():
            v_feat = clip_model.encode_image(videos.view(-1, 3, 224, 224)).view(b, 16, -1).to(torch.bfloat16)

            a_feat = clap_model.get_audio_embedding_from_data(x=audios, use_tensor=True)
            if isinstance(a_feat, np.ndarray):
                a_feat = torch.from_numpy(a_feat).to(device)
            a_feat = a_feat.unsqueeze(1).to(torch.bfloat16)

        txt_inputs = model.tokenizer(
            transcriptions,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        ).to(device)

        preds = model(
            input_ids=txt_inputs.input_ids,
            attention_mask=txt_inputs.attention_mask,
            video_features=v_feat,
            audio_features=a_feat
        )

        loss = criterion(preds, target_labels)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss_sum += loss.item()
        pbar.set_postfix({"Train_MAE": f"{loss.item():.4f}", "Best_Val_1-MAE": f"{best_val_1mae:.4f}"})

    avg_train_mae = train_loss_sum / max(1, len(train_loader))
    avg_train_1mae = 1.0 - avg_train_mae

    # =========================
    # Val
    # =========================
    model.eval()
    val_loss_sum = 0.0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="🛡️ [VAL]"):
            videos = batch["video"].to(device)
            audios = batch["audio"].to(device).float()
            target_labels = batch["label"].to(device).float()
            transcriptions = batch["transcription"]
            b = videos.shape[0]

            v_feat = clip_model.encode_image(videos.view(-1, 3, 224, 224)).view(b, 16, -1).to(torch.bfloat16)

            a_feat = clap_model.get_audio_embedding_from_data(x=audios, use_tensor=True)
            if isinstance(a_feat, np.ndarray):
                a_feat = torch.from_numpy(a_feat).to(device)
            a_feat = a_feat.unsqueeze(1).to(torch.bfloat16)

            txt_inputs = model.tokenizer(
                transcriptions,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=128
            ).to(device)

            preds = model(
                input_ids=txt_inputs.input_ids,
                attention_mask=txt_inputs.attention_mask,
                video_features=v_feat,
                audio_features=a_feat
            )

            val_loss_sum += criterion(preds, target_labels).item()

    avg_val_mae = val_loss_sum / max(1, len(val_loader))
    avg_val_1mae = 1.0 - avg_val_mae

    history["train_mae"].append(avg_train_mae)
    history["val_mae"].append(avg_val_mae)
    history["train_1mae"].append(avg_train_1mae)
    history["val_1mae"].append(avg_val_1mae)

    # 每轮都打印目前最好的 1-MAE
    print(f"📊 Ep {epoch+1} | Train MAE: {avg_train_mae:.4f} (1-MAE={avg_train_1mae:.4f}) "
          f"| Val MAE: {avg_val_mae:.4f} (1-MAE={avg_val_1mae:.4f}) "
          f"| Best Val 1-MAE: {best_val_1mae:.4f}")

    # ✅ 越大越好：保存
    if avg_val_1mae > best_val_1mae:
        best_val_1mae = avg_val_1mae
        torch.save(model.state_dict(), "best_latent_cot_gemma_model.pt")
        print(f"🏆 New Best Model Saved! Best Val 1-MAE = {best_val_1mae:.4f}\n")


🚀 Latent CoT Personality (Gemma) 모델 초기화 중...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Load our best checkpoint in the paper.
The checkpoint is already downloaded
Load Checkpoint...
logit_scale_a 	 Loaded
logit_scale_t 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_real.weight 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_imag.weight 	 Loaded
audio_branch.logmel_extractor.melW 	 Loaded
audio_branch.bn0.weight 	 Loaded
audio_branch.bn0.bias 	 Loaded
audio_branch.patch_embed.proj.weight 	 Loaded
audio_branch.patch_embed.proj.bias 	 Loaded
audio_branch.patch_embed.norm.weight 	 Loaded
audio_branch.patch_embed.norm.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm1.weight 	 Loaded
audio_branch.layers.0.blocks.0.norm1.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.relative_position_bias_table 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm2.we

... (56 outputs left out)

🔥 [Latent-CoT|Gemma] Ep 29/30: 100%|██████████| 1031/1031 [06:48<00:00,  2.52it/s, Train_MAE=0.0161, Best_Val_1-MAE=0.9116]
🛡️ [VAL]: 100%|██████████| 337/337 [01:19<00:00,  4.25it/s]


📊 Ep 29 | Train MAE: 0.0342 (1-MAE=0.9658) | Val MAE: 0.0994 (1-MAE=0.9006) | Best Val 1-MAE: 0.9116


🔥 [Latent-CoT|Gemma] Ep 30/30: 100%|██████████| 1031/1031 [06:47<00:00,  2.53it/s, Train_MAE=0.0279, Best_Val_1-MAE=0.9116]
🛡️ [VAL]: 100%|██████████| 337/337 [01:19<00:00,  4.25it/s]

📊 Ep 30 | Train MAE: 0.0340 (1-MAE=0.9660) | Val MAE: 0.0971 (1-MAE=0.9029) | Best Val 1-MAE: 0.9116
